# Task 1: Data exploration

## 1. Load all 15 tables

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("data.duckdb", read_only=True)
TODAY = pd.Timestamp("2026-05-04")

In [2]:
# transaction tables: one row per event (order, delivery, invoice, payment)
vbak = con.sql("select * from src_vbak").df()   # order headers
vbap = con.sql("select * from src_vbap").df()   # order lines
likp = con.sql("select * from src_likp").df()   # delivery headers
lips = con.sql("select * from src_lips").df()   # delivery lines
vbrk = con.sql("select * from src_vbrk").df()   # billing headers
vbrp = con.sql("select * from src_vbrp").df()   # billing lines
bsid = con.sql("select * from src_bsid").df()   # open receivables
bsad = con.sql("select * from src_bsad").df()   # cleared receivables

# lookup tables: who, what, where
kna1  = con.sql("select * from src_kna1").df()   # customers
mara  = con.sql("select * from src_mara").df()   # materials
makt  = con.sql("select * from src_makt").df()   # material names
t001w = con.sql("select * from src_t001w").df()  # plants
tvkot = con.sql("select * from src_tvkot").df()  # sales orgs
tvtwt = con.sql("select * from src_tvtwt").df()  # channels
tvagt = con.sql("select * from src_tvagt").df()  # rejection reasons

## 2. Each table
`info()` = column types and non-empty counts. `describe(include='all')` (transaction tables) = count, distinct values (`unique`), most common value (`top`), min, mean and max. `duplicated(key).any()` = False if the key never repeats (pandas shows it as `np.False_`) (keys are not enforced, schema lines 9-10).

### vbak (order headers)

In [3]:
vbak

,vbeln,erdat,auart,kunnr,vkorg,vtweg,spart,netwr,waerk,vdatu
0,4500000001,2025-10-01,TA,C0000001,EU01,10,HA,3400.0,EUR,2025-10-15
1,4500000002,2025-10-05,TA,C0000001,EU01,10,AD,3800.0,EUR,2025-10-20
2,4500000003,2025-10-10,TA,C0000003,EU01,10,HA,1250.0,EUR,2025-10-25
3,4500000004,2026-03-15,TA,C0000003,EU01,10,HA,4350.0,EUR,2026-04-01
4,4500000005,2025-11-01,TA,C0000002,NA01,10,AD,6000.0,USD,2025-11-20
5,4500000006,2025-10-20,KR,C0000001,EU01,10,HA,-500.0,EUR,NaT
6,4500000007,2026-01-10,TA,C0000005,EU01,10,HA,8200.0,EUR,2026-02-01
7,4500000008,2026-04-15,TA,C0000001,EU01,10,HA,2000.0,EUR,2026-05-01


In [4]:
vbak.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   vbeln   8 non-null      object        
 1   erdat   8 non-null      datetime64[us]
 2   auart   8 non-null      object        
 3   kunnr   8 non-null      object        
 4   vkorg   8 non-null      object        
 5   vtweg   8 non-null      object        
 6   spart   8 non-null      object        
 7   netwr   8 non-null      float64       
 8   waerk   8 non-null      object        
 9   vdatu   7 non-null      datetime64[us]
dtypes: datetime64[us](2), float64(1), object(7)
memory usage: 768.0+ bytes


In [5]:
vbak.describe(include='all')

,vbeln,erdat,auart,kunnr,vkorg,vtweg,spart,netwr,waerk,vdatu
count,8,8,8,8,8,8,8,8.00000,8,7
unique,8,NaN,2,4,2,1,2,NaN,2,NaN
top,4500000001,NaN,TA,C0000001,EU01,10,HA,NaN,EUR,NaN
freq,1,NaN,7,4,7,8,6,NaN,7,NaN
mean,NaN,2025-12-05 15:00:00,NaN,NaN,NaN,NaN,NaN,3562.50000,NaN,2025-12-29 03:25:42.857142
min,NaN,2025-10-01 00:00:00,NaN,NaN,NaN,NaN,NaN,-500.00000,NaN,2025-10-15 00:00:00
25%,NaN,2025-10-08 18:00:00,NaN,NaN,NaN,NaN,NaN,1812.50000,NaN,2025-10-22 12:00:00
50%,NaN,2025-10-26 00:00:00,NaN,NaN,NaN,NaN,NaN,3600.00000,NaN,2025-11-20 00:00:00
75%,NaN,2026-01-26 00:00:00,NaN,NaN,NaN,NaN,NaN,4762.50000,NaN,2026-03-02 12:00:00
max,NaN,2026-04-15 00:00:00,NaN,NaN,NaN,NaN,NaN,8200.00000,NaN,2026-05-01 00:00:00


In [6]:
vbak.duplicated(['vbeln']).any()   # any repeated key? False = unique

np.False_

### vbap (order lines)

In [7]:
vbap

,vbeln,posnr,matnr,kwmeng,meins,netpr,waerk,werks,pstyv,abgru
0,4500000001,000010,MAT00001,100.0,EA,25.0,EUR,DE01,TAN,None
1,4500000001,000020,MAT00005,50.0,EA,18.0,EUR,DE01,TAN,None
2,4500000002,000010,MAT00003,200.0,KG,15.0,EUR,DE01,TAN,None
3,4500000002,000020,MAT00004,100.0,L,8.0,EUR,DE01,TAN,None
4,4500000003,000010,MAT00001,50.0,EA,25.0,EUR,DE01,TAN,None
5,4500000003,000020,MAT00002,30.0,EA,42.0,EUR,DE01,TAN,01
6,4500000004,000010,MAT00005,200.0,EA,18.0,EUR,DE01,TAN,None
7,4500000004,000020,MAT00003,50.0,KG,15.0,EUR,DE01,TAN,None
8,4500000005,000010,MAT00003,500.0,KG,12.0,USD,US01,TAN,None
9,4500000006,000010,MAT00001,20.0,EA,-25.0,EUR,DE01,KRN,None


In [8]:
vbap.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   vbeln   13 non-null     object 
 1   posnr   13 non-null     object 
 2   matnr   13 non-null     object 
 3   kwmeng  13 non-null     float64
 4   meins   13 non-null     object 
 5   netpr   13 non-null     float64
 6   waerk   13 non-null     object 
 7   werks   13 non-null     object 
 8   pstyv   13 non-null     object 
 9   abgru   1 non-null      object 
dtypes: float64(2), object(8)
memory usage: 1.1+ KB


In [9]:
vbap.describe(include='all')

,vbeln,posnr,matnr,kwmeng,meins,netpr,waerk,werks,pstyv,abgru
count,13,13,13,13.000000,13,13.000000,13,13,13,1
unique,8,2,5,NaN,3,NaN,2,3,2,1
top,4500000001,000010,MAT00001,NaN,EA,NaN,EUR,DE01,TAN,01
freq,2,8,4,NaN,8,NaN,12,10,12,1
mean,NaN,NaN,NaN,152.307692,NaN,17.538462,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,164.223113,NaN,16.830604,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,20.000000,NaN,-25.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,50.000000,NaN,12.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,100.000000,NaN,18.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,200.000000,NaN,25.000000,NaN,NaN,NaN,NaN


In [10]:
vbap.duplicated(['vbeln', 'posnr']).any()   # any repeated key? False = unique

np.False_

In [11]:
int((vbap['abgru'] == '').sum())   # blank rejection reasons ('' also means active, schema line 164)

0

### likp (delivery headers)

In [12]:
likp

,vbeln,erdat,wadat_ist,kunnr,vstel
0,8000000001,2025-10-11,2025-10-13,C0000001,SH01
1,8000000002,2025-10-16,2025-10-18,C0000001,SH01
2,8000000003,2025-11-06,2025-11-08,C0000001,SH01
3,8000000004,2025-10-26,2025-10-28,C0000003,SH01
4,8000000005,2025-11-15,2025-11-17,C0000002,SH02
5,8000000006,2026-01-27,2026-01-29,C0000005,SH01
6,8000000007,2026-04-23,2026-04-25,C0000001,SH01


In [13]:
likp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   vbeln      7 non-null      object        
 1   erdat      7 non-null      datetime64[us]
 2   wadat_ist  7 non-null      datetime64[us]
 3   kunnr      7 non-null      object        
 4   vstel      7 non-null      object        
dtypes: datetime64[us](2), object(3)
memory usage: 408.0+ bytes


In [14]:
likp.describe(include='all')

,vbeln,erdat,wadat_ist,kunnr,vstel
count,7,7,7,7,7
unique,7,NaN,NaN,4,2
top,8000000001,NaN,NaN,C0000001,SH01
freq,1,NaN,NaN,4,6
mean,NaN,2025-12-04 17:08:34.285714,2025-12-06 17:08:34.285714,NaN,NaN
min,NaN,2025-10-11 00:00:00,2025-10-13 00:00:00,NaN,NaN
25%,NaN,2025-10-21 00:00:00,2025-10-23 00:00:00,NaN,NaN
50%,NaN,2025-11-06 00:00:00,2025-11-08 00:00:00,NaN,NaN
75%,NaN,2025-12-21 12:00:00,2025-12-23 12:00:00,NaN,NaN
max,NaN,2026-04-23 00:00:00,2026-04-25 00:00:00,NaN,NaN


In [15]:
likp.duplicated(['vbeln']).any()   # any repeated key? False = unique

np.False_

### lips (delivery lines)

In [16]:
lips

,vbeln,posnr,matnr,lfimg,meins,werks,vgbel,vgpos
0,8000000001,000010,MAT00001,100.0,EA,DE01,4500000001,000010
1,8000000001,000020,MAT00005,50.0,EA,DE01,4500000001,000020
2,8000000002,000010,MAT00003,100.0,KG,DE01,4500000002,000010
3,8000000002,000020,MAT00004,100.0,L,DE01,4500000002,000020
4,8000000003,000010,MAT00003,100.0,KG,DE01,4500000002,000010
5,8000000004,000010,MAT00001,50.0,EA,DE01,4500000003,000010
6,8000000005,000010,MAT00003,500.0,KG,US01,4500000005,000010
7,8000000006,000010,MAT00004,500.0,L,DE02,4500000007,000010
8,8000000006,000020,MAT00002,100.0,EA,DE02,4500000007,000020
9,8000000007,000010,MAT00001,80.0,EA,DE01,4500000008,000010


In [17]:
lips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   vbeln   10 non-null     object 
 1   posnr   10 non-null     object 
 2   matnr   10 non-null     object 
 3   lfimg   10 non-null     float64
 4   meins   10 non-null     object 
 5   werks   10 non-null     object 
 6   vgbel   10 non-null     object 
 7   vgpos   10 non-null     object 
dtypes: float64(1), object(7)
memory usage: 768.0+ bytes


In [18]:
lips.describe(include='all')

,vbeln,posnr,matnr,lfimg,meins,werks,vgbel,vgpos
count,10,10,10,10.000000,10,10,10,10
unique,7,2,5,NaN,3,3,6,2
top,8000000001,000010,MAT00001,NaN,EA,DE01,4500000002,000010
freq,2,7,3,NaN,5,7,3,7
mean,NaN,NaN,NaN,168.000000,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,176.118647,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,50.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,85.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,100.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,100.000000,NaN,NaN,NaN,NaN


In [19]:
lips.duplicated(['vbeln', 'posnr']).any()   # any repeated key? False = unique

np.False_

### vbrk (billing headers)

In [20]:
vbrk

,vbeln,erdat,fkart,kunrg,waerk,netwr,mwsbp,fkdat,vkorg
0,9000000001,2025-10-14,F2,C0000001,EUR,3400.0,646.0,2025-10-14,EU01
1,9000000002,2025-10-19,F2,C0000001,EUR,2300.0,437.0,2025-10-19,EU01
2,9000000003,2025-11-10,F2,C0000001,EUR,1500.0,285.0,2025-11-10,EU01
3,9000000004,2025-10-29,F2,C0000003,EUR,1250.0,237.5,2025-10-29,EU01
4,9000000005,2025-11-18,F2,C0000002,USD,6000.0,0.0,2025-11-18,NA01
5,9000000006,2025-10-22,G2,C0000001,EUR,-500.0,-95.0,2025-10-22,EU01
6,9000000007,2026-01-30,F2,C0000005,EUR,4000.0,760.0,2026-01-30,EU01
7,9000000008,2026-04-26,F2,C0000001,EUR,2000.0,380.0,2026-04-26,EU01


In [21]:
vbrk.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   vbeln   8 non-null      object        
 1   erdat   8 non-null      datetime64[us]
 2   fkart   8 non-null      object        
 3   kunrg   8 non-null      object        
 4   waerk   8 non-null      object        
 5   netwr   8 non-null      float64       
 6   mwsbp   8 non-null      float64       
 7   fkdat   8 non-null      datetime64[us]
 8   vkorg   8 non-null      object        
dtypes: datetime64[us](2), float64(2), object(5)
memory usage: 704.0+ bytes


In [22]:
vbrk.describe(include='all')

,vbeln,erdat,fkart,kunrg,waerk,netwr,mwsbp,fkdat,vkorg
count,8,8,8,8,8,8.000000,8.000000,8,8
unique,8,NaN,2,4,2,NaN,NaN,NaN,2
top,9000000001,NaN,F2,C0000001,EUR,NaN,NaN,NaN,EU01
freq,1,NaN,7,5,7,NaN,NaN,NaN,7
mean,NaN,2025-12-02 00:00:00,NaN,NaN,NaN,2493.750000,331.312500,2025-12-02 00:00:00,NaN
min,NaN,2025-10-14 00:00:00,NaN,NaN,NaN,-500.000000,-95.000000,2025-10-14 00:00:00,NaN
25%,NaN,2025-10-21 06:00:00,NaN,NaN,NaN,1437.500000,178.125000,2025-10-21 06:00:00,NaN
50%,NaN,2025-11-04 00:00:00,NaN,NaN,NaN,2150.000000,332.500000,2025-11-04 00:00:00,NaN
75%,NaN,2025-12-06 06:00:00,NaN,NaN,NaN,3550.000000,489.250000,2025-12-06 06:00:00,NaN
max,NaN,2026-04-26 00:00:00,NaN,NaN,NaN,6000.000000,760.000000,2026-04-26 00:00:00,NaN


In [23]:
vbrk.duplicated(['vbeln']).any()   # any repeated key? False = unique

np.False_

### vbrp (billing lines)

In [24]:
vbrp

,vbeln,posnr,matnr,fkimg,meins,netwr,aubel,aupos,vgbel,vgpos
0,9000000001,000010,MAT00001,100.0,EA,2500.0,4500000001,000010,8000000001,000010
1,9000000001,000020,MAT00005,50.0,EA,900.0,4500000001,000020,8000000001,000020
2,9000000002,000010,MAT00003,100.0,KG,1500.0,4500000002,000010,8000000002,000010
3,9000000002,000020,MAT00004,100.0,L,800.0,4500000002,000020,8000000002,000020
4,9000000003,000010,MAT00003,100.0,KG,1500.0,4500000002,000010,8000000003,000010
5,9000000004,000010,MAT00001,50.0,EA,1250.0,4500000003,000010,8000000004,000010
6,9000000005,000010,MAT00003,500.0,KG,6000.0,4500000005,000010,8000000005,000010
7,9000000006,000010,MAT00001,20.0,EA,-500.0,4500000006,000010,9000000001,000010
8,9000000007,000010,MAT00004,500.0,L,4000.0,4500000007,000010,8000000006,000010
9,9000000008,000010,MAT00001,80.0,EA,2000.0,4500000008,000010,8000000007,000010


In [25]:
vbrp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   vbeln   10 non-null     object 
 1   posnr   10 non-null     object 
 2   matnr   10 non-null     object 
 3   fkimg   10 non-null     float64
 4   meins   10 non-null     object 
 5   netwr   10 non-null     float64
 6   aubel   10 non-null     object 
 7   aupos   10 non-null     object 
 8   vgbel   10 non-null     object 
 9   vgpos   10 non-null     object 
dtypes: float64(2), object(8)
memory usage: 928.0+ bytes


In [26]:
vbrp.describe(include='all')

,vbeln,posnr,matnr,fkimg,meins,netwr,aubel,aupos,vgbel,vgpos
count,10,10,10,10.000000,10,10.000000,10,10,10,10
unique,8,2,4,NaN,3,NaN,7,2,8,2
top,9000000001,000010,MAT00001,NaN,EA,NaN,4500000002,000010,8000000001,000010
freq,2,8,4,NaN,5,NaN,3,8,2,8
mean,NaN,NaN,NaN,160.000000,NaN,1995.000000,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,181.291662,NaN,1832.189037,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,20.000000,NaN,-500.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,57.500000,NaN,987.500000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,100.000000,NaN,1500.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,100.000000,NaN,2375.000000,NaN,NaN,NaN,NaN


In [27]:
vbrp.duplicated(['vbeln', 'posnr']).any()   # any repeated key? False = unique

np.False_

### bsid (open receivables)

In [28]:
bsid

,bukrs,kunnr,belnr,gjahr,buzei,bldat,faedt,wrbtr,waers,rebzg
0,EU01,C0000001,1400000003,2025,001,2025-11-10,2025-12-10,1500.0,EUR,9000000003
1,EU01,C0000005,1400000007,2026,001,2026-01-30,2026-03-31,4000.0,EUR,9000000007
2,EU01,C0000001,1400000008,2026,001,2026-04-26,2026-05-26,2000.0,EUR,9000000008


In [29]:
bsid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   bukrs   3 non-null      object        
 1   kunnr   3 non-null      object        
 2   belnr   3 non-null      object        
 3   gjahr   3 non-null      int32         
 4   buzei   3 non-null      object        
 5   bldat   3 non-null      datetime64[us]
 6   faedt   3 non-null      datetime64[us]
 7   wrbtr   3 non-null      float64       
 8   waers   3 non-null      object        
 9   rebzg   3 non-null      object        
dtypes: datetime64[us](2), float64(1), int32(1), object(6)
memory usage: 356.0+ bytes


In [30]:
bsid.describe(include='all')

,bukrs,kunnr,belnr,gjahr,buzei,bldat,faedt,wrbtr,waers,rebzg
count,3,3,3,3.000000,3,3,3,3.000000,3,3
unique,1,2,3,NaN,1,NaN,NaN,NaN,1,3
top,EU01,C0000001,1400000003,NaN,001,NaN,NaN,NaN,EUR,9000000003
freq,3,2,1,NaN,3,NaN,NaN,NaN,3,1
mean,NaN,NaN,NaN,2025.666667,NaN,2026-01-31 16:00:00,2026-03-12 16:00:00,2500.000000,NaN,NaN
min,NaN,NaN,NaN,2025.000000,NaN,2025-11-10 00:00:00,2025-12-10 00:00:00,1500.000000,NaN,NaN
25%,NaN,NaN,NaN,2025.500000,NaN,2025-12-20 12:00:00,2026-02-03 12:00:00,1750.000000,NaN,NaN
50%,NaN,NaN,NaN,2026.000000,NaN,2026-01-30 00:00:00,2026-03-31 00:00:00,2000.000000,NaN,NaN
75%,NaN,NaN,NaN,2026.000000,NaN,2026-03-14 00:00:00,2026-04-28 00:00:00,3000.000000,NaN,NaN
max,NaN,NaN,NaN,2026.000000,NaN,2026-04-26 00:00:00,2026-05-26 00:00:00,4000.000000,NaN,NaN


In [31]:
bsid.duplicated(['bukrs', 'belnr', 'gjahr', 'buzei']).any()   # any repeated key? False = unique

np.False_

### bsad (cleared receivables)

In [32]:
bsad

,bukrs,kunnr,belnr,gjahr,buzei,bldat,faedt,augdt,wrbtr,waers,rebzg
0,EU01,C0000001,1400000001,2025,001,2025-10-14,2025-11-13,2025-11-05,3400.0,EUR,9000000001
1,EU01,C0000001,1400000002,2025,001,2025-10-19,2025-11-18,2025-11-15,2300.0,EUR,9000000002
2,EU01,C0000003,1400000004,2025,001,2025-10-29,2025-11-28,2025-11-20,1250.0,EUR,9000000004
3,NA01,C0000002,1400000005,2025,001,2025-11-18,2025-12-18,2025-12-10,6000.0,USD,9000000005
4,EU01,C0000001,1400000006,2025,001,2025-10-22,2025-11-21,2025-11-05,-500.0,EUR,9000000006


In [33]:
bsad.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   bukrs   5 non-null      object        
 1   kunnr   5 non-null      object        
 2   belnr   5 non-null      object        
 3   gjahr   5 non-null      int32         
 4   buzei   5 non-null      object        
 5   bldat   5 non-null      datetime64[us]
 6   faedt   5 non-null      datetime64[us]
 7   augdt   5 non-null      datetime64[us]
 8   wrbtr   5 non-null      float64       
 9   waers   5 non-null      object        
 10  rebzg   5 non-null      object        
dtypes: datetime64[us](3), float64(1), int32(1), object(6)
memory usage: 548.0+ bytes


In [34]:
bsad.describe(include='all')

,bukrs,kunnr,belnr,gjahr,buzei,bldat,faedt,augdt,wrbtr,waers,rebzg
count,5,5,5,5.0,5,5,5,5,5.000000,5,5
unique,2,3,5,NaN,1,NaN,NaN,NaN,NaN,2,5
top,EU01,C0000001,1400000001,NaN,001,NaN,NaN,NaN,NaN,EUR,9000000001
freq,4,3,1,NaN,5,NaN,NaN,NaN,NaN,4,1
mean,NaN,NaN,NaN,2025.0,NaN,2025-10-26 14:24:00,2025-11-25 14:24:00,2025-11-17 00:00:00,2490.000000,NaN,NaN
min,NaN,NaN,NaN,2025.0,NaN,2025-10-14 00:00:00,2025-11-13 00:00:00,2025-11-05 00:00:00,-500.000000,NaN,NaN
25%,NaN,NaN,NaN,2025.0,NaN,2025-10-19 00:00:00,2025-11-18 00:00:00,2025-11-05 00:00:00,1250.000000,NaN,NaN
50%,NaN,NaN,NaN,2025.0,NaN,2025-10-22 00:00:00,2025-11-21 00:00:00,2025-11-15 00:00:00,2300.000000,NaN,NaN
75%,NaN,NaN,NaN,2025.0,NaN,2025-10-29 00:00:00,2025-11-28 00:00:00,2025-11-20 00:00:00,3400.000000,NaN,NaN
max,NaN,NaN,NaN,2025.0,NaN,2025-11-18 00:00:00,2025-12-18 00:00:00,2025-12-10 00:00:00,6000.000000,NaN,NaN


In [35]:
bsad.duplicated(['bukrs', 'belnr', 'gjahr', 'buzei']).any()   # any repeated key? False = unique

np.False_

### kna1 (customers)

In [36]:
kna1

,kunnr,name1,land1,regio,ktokd
0,C0000001,Arcturus Professional GmbH,DE,NW,GP01
1,C0000002,SalonCorp USA Inc.,US,NJ,GP01
2,C0000003,Beaute Pro SARL,FR,IDF,GP01
3,C0000004,ProCare Italia SpA,IT,LOM,GP01
4,C0000005,Nordic Hair Solutions AB,SE,26,GP01


In [37]:
kna1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   kunnr   5 non-null      object
 1   name1   5 non-null      object
 2   land1   5 non-null      object
 3   regio   5 non-null      object
 4   ktokd   5 non-null      object
dtypes: object(5)
memory usage: 328.0+ bytes


In [38]:
kna1.duplicated(['kunnr']).any()   # any repeated key? False = unique

np.False_

### mara (materials)

In [39]:
mara

,matnr,matkl,meins,mtart
0,MAT00001,HAIRPRO,EA,FERT
1,MAT00002,HAIRPRO,EA,FERT
2,MAT00003,ADHESIV,KG,FERT
3,MAT00004,ADHESIV,L,FERT
4,MAT00005,HAIRPRO,EA,FERT


In [40]:
mara.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   matnr   5 non-null      object
 1   matkl   5 non-null      object
 2   meins   5 non-null      object
 3   mtart   5 non-null      object
dtypes: object(4)
memory usage: 288.0+ bytes


In [41]:
mara.duplicated(['matnr']).any()   # any repeated key? False = unique

np.False_

### makt (material names)

In [42]:
makt

,matnr,spras,maktx
0,MAT00001,EN,Professional Shampoo 1L
1,MAT00002,EN,Color Treatment Professional
2,MAT00003,EN,Adhesive Bond XT
3,MAT00004,EN,Surface Prep Liquid
4,MAT00005,EN,Conditioner Pro 1L


In [43]:
makt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   matnr   5 non-null      object
 1   spras   5 non-null      object
 2   maktx   5 non-null      object
dtypes: object(3)
memory usage: 248.0+ bytes


In [44]:
makt.duplicated(['matnr', 'spras']).any()   # any repeated key? False = unique

np.False_

### t001w (plants)

In [45]:
t001w

,werks,name1,land1
0,DE01,Duesseldorf Warehouse,DE
1,DE02,Hamburg Distribution Center,DE
2,US01,New Jersey Fulfillment Center,US


In [46]:
t001w.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   werks   3 non-null      object
 1   name1   3 non-null      object
 2   land1   3 non-null      object
dtypes: object(3)
memory usage: 200.0+ bytes


In [47]:
t001w.duplicated(['werks']).any()   # any repeated key? False = unique

np.False_

### tvkot (sales orgs)

In [48]:
tvkot

,vkorg,vtext
0,EU01,Europe Sales Organization
1,NA01,North America Sales Organization


In [49]:
tvkot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   vkorg   2 non-null      object
 1   vtext   2 non-null      object
dtypes: object(2)
memory usage: 160.0+ bytes


In [50]:
tvkot.duplicated(['vkorg']).any()   # any repeated key? False = unique

np.False_

### tvtwt (channels)

In [51]:
tvtwt

,vtweg,vtext
0,10,Direct Sales
1,20,Retail / Wholesale


In [52]:
tvtwt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   vtweg   2 non-null      object
 1   vtext   2 non-null      object
dtypes: object(2)
memory usage: 160.0+ bytes


In [53]:
tvtwt.duplicated(['vtweg']).any()   # any repeated key? False = unique

np.False_

### tvagt (rejection reasons)

In [54]:
tvagt

,abgru,bezei
0,01,Cancelled by customer
1,02,Out of stock
2,03,Pricing dispute


In [55]:
tvagt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   abgru   3 non-null      object
 1   bezei   3 non-null      object
dtypes: object(2)
memory usage: 176.0+ bytes


In [56]:
tvagt.duplicated(['abgru']).any()   # any repeated key? False = unique

np.False_

## 3. Links between tables

Each cell lists child rows (`c`) whose parent (`p`) is missing. An empty table = the link is clean.

In [57]:
# order line -> order
con.sql("""
    select c.*
    from src_vbap c
    left join src_vbak p on p.vbeln = c.vbeln
    where p.vbeln is null
""").df()

,vbeln,posnr,matnr,kwmeng,meins,netpr,waerk,werks,pstyv,abgru


In [58]:
# delivery line -> delivery
con.sql("""
    select c.*
    from src_lips c
    left join src_likp p on p.vbeln = c.vbeln
    where p.vbeln is null
""").df()

,vbeln,posnr,matnr,lfimg,meins,werks,vgbel,vgpos


In [59]:
# delivery line -> order line
con.sql("""
    select c.*
    from src_lips c
    left join src_vbap p on p.vbeln = c.vgbel and p.posnr = c.vgpos
    where p.vbeln is null
""").df()

,vbeln,posnr,matnr,lfimg,meins,werks,vgbel,vgpos


In [60]:
# billing line -> billing document
con.sql("""
    select c.*
    from src_vbrp c
    left join src_vbrk p on p.vbeln = c.vbeln
    where p.vbeln is null
""").df()

,vbeln,posnr,matnr,fkimg,meins,netwr,aubel,aupos,vgbel,vgpos


In [61]:
# billing line -> order line
con.sql("""
    select c.*
    from src_vbrp c
    left join src_vbap p on p.vbeln = c.aubel and p.posnr = c.aupos
    where p.vbeln is null
""").df()

,vbeln,posnr,matnr,fkimg,meins,netwr,aubel,aupos,vgbel,vgpos


In [62]:
# open receivable -> billing document
con.sql("""
    select c.*
    from src_bsid c
    left join src_vbrk p on p.vbeln = c.rebzg
    where p.vbeln is null
""").df()

,bukrs,kunnr,belnr,gjahr,buzei,bldat,faedt,wrbtr,waers,rebzg


In [63]:
# cleared receivable -> billing document
con.sql("""
    select c.*
    from src_bsad c
    left join src_vbrk p on p.vbeln = c.rebzg
    where p.vbeln is null
""").df()

,bukrs,kunnr,belnr,gjahr,buzei,bldat,faedt,augdt,wrbtr,waers,rebzg


In [64]:
# invoice line (F2) -> delivery line it bills (vgbel/vgpos, schema lines 60-63)
con.sql("""
    select c.*
    from src_vbrp c
    join src_vbrk k on k.vbeln = c.vbeln and k.fkart = 'F2'
    left join src_lips p on p.vbeln = c.vgbel and p.posnr = c.vgpos
    where p.vbeln is null
""").df()

,vbeln,posnr,matnr,fkimg,meins,netwr,aubel,aupos,vgbel,vgpos


In [65]:
# credit memo line (G2) -> original invoice line (vgbel/vgpos, schema lines 60-63)
con.sql("""
    select c.*
    from src_vbrp c
    join src_vbrk k on k.vbeln = c.vbeln and k.fkart = 'G2'
    left join src_vbrp p on p.vbeln = c.vgbel and p.posnr = c.vgpos
    where p.vbeln is null
""").df()

,vbeln,posnr,matnr,fkimg,meins,netwr,aubel,aupos,vgbel,vgpos


In [66]:
# delivery lines per order line
lips.groupby(['vgbel', 'vgpos']).size()

vgbel       vgpos 
4500000001  000010    1
            000020    1
4500000002  000010    2
            000020    1
4500000003  000010    1
4500000005  000010    1
4500000007  000010    1
            000020    1
4500000008  000010    1
dtype: int64

In [67]:
# billing lines per order line
vbrp.groupby(['aubel', 'aupos']).size()

aubel       aupos 
4500000001  000010    1
            000020    1
4500000002  000010    2
            000020    1
4500000003  000010    1
4500000005  000010    1
4500000006  000010    1
4500000007  000010    1
4500000008  000010    1
dtype: int64

In [68]:
# order line 4500000002/000010 joined directly to its delivery lines and billing lines
d = lips[(lips['vgbel'] == '4500000002') & (lips['vgpos'] == '000010')]
b = vbrp[(vbrp['aubel'] == '4500000002') & (vbrp['aupos'] == '000010')]
len(d.merge(b, left_on=['vgbel', 'vgpos'], right_on=['aubel', 'aupos']))   # 4 rows: the line's 3,000 counted 4 times = 12,000

4

In [69]:
# billed value after that join (true: 3,000)
float(d.merge(b, left_on=['vgbel', 'vgpos'], right_on=['aubel', 'aupos'])['netwr'].sum())

6000.0

In [70]:
# the order line joined straight to its deliveries and billing lines: its 3,000 is summed 4 times
con.sql("""
    select count(*) as rows, sum(o.kwmeng * o.netpr) as ordered_value
    from src_vbap o
    join src_lips l on l.vgbel = o.vbeln and l.vgpos = o.posnr
    join src_vbrp b on b.aubel = o.vbeln and b.aupos = o.posnr
    where o.vbeln = '4500000002' and o.posnr = '000010'
""").df()

,rows,ordered_value
0,4,12000.0


In [71]:
# what vgbel points to, per billing type
lines = vbrp.merge(vbrk[['vbeln', 'fkart']], on='vbeln')   # add the billing type to each billing line
lines[['vbeln', 'fkart', 'vgbel']]

,vbeln,fkart,vgbel
0,9000000001,F2,8000000001
1,9000000001,F2,8000000001
2,9000000002,F2,8000000002
3,9000000002,F2,8000000002
4,9000000003,F2,8000000003
5,9000000004,F2,8000000004
6,9000000005,F2,8000000005
7,9000000006,G2,9000000001
8,9000000007,F2,8000000006
9,9000000008,F2,8000000007


In [72]:
# delivered but never billed: delivery lines that no invoice (F2) line points to (vgbel/vgpos, schema line 60)
con.sql("""
    select l.vbeln, l.posnr, l.vgbel, l.vgpos, l.lfimg, o.netpr, l.lfimg * o.netpr as value
    from src_lips l
    join src_vbap o on o.vbeln = l.vgbel and o.posnr = l.vgpos
    left join src_vbrp b on b.vgbel = l.vbeln and b.vgpos = l.posnr
        and b.vbeln in (select vbeln from src_vbrk where fkart = 'F2')
    where b.vbeln is null
""").df()

,vbeln,posnr,vgbel,vgpos,lfimg,netpr,value
0,8000000006,000020,4500000007,000020,100.0,42.0,4200.0


In [73]:
# columns in bsad but not in bsid
bsad.columns.difference(bsid.columns)

Index(['augdt'], dtype='object')

In [74]:
# same receivable in both bsid and bsad? (schema lines 246-248)
key = ['bukrs', 'belnr', 'gjahr', 'buzei']   # the 4 columns that name one receivable
bsid[key].merge(bsad[key])                   # keys found in both tables (empty = none)

,bukrs,belnr,gjahr,buzei


In [75]:
# billing documents with no receivable?
ar = pd.concat([bsid, bsad], ignore_index=True)   # all receivables in one table (open + cleared), numbered 0-7
has_ar = vbrk['vbeln'].isin(ar['rebzg'])    # True = this billing doc has a receivable
vbrk[has_ar == False]                       # show the ones without (empty = none)

,vbeln,erdat,fkart,kunrg,waerk,netwr,mwsbp,fkdat,vkorg


## 4. Totals: header vs lines

In [76]:
# value of each order line
vbap['value'] = vbap['kwmeng'] * vbap['netpr']
active = vbap[vbap['abgru'].fillna('') == '']        # active lines: no rejection reason (NULL or '')

In [77]:
# orders: header value vs the sum of its active lines
pd.DataFrame({'header': vbak.set_index('vbeln')['netwr'], 'active_lines': active.groupby('vbeln')['value'].sum()})

,header,active_lines
vbeln,,
4500000001,3400.0,3400.0
4500000002,3800.0,3800.0
4500000003,1250.0,1250.0
4500000004,4350.0,4350.0
4500000005,6000.0,6000.0
4500000006,-500.0,-500.0
4500000007,8200.0,8200.0
4500000008,2000.0,2000.0


In [78]:
# billing documents: header value vs the sum of its lines
pd.DataFrame({'header': vbrk.set_index('vbeln')['netwr'], 'lines': vbrp.groupby('vbeln')['netwr'].sum()})

,header,lines
vbeln,,
9000000001,3400.0,3400.0
9000000002,2300.0,2300.0
9000000003,1500.0,1500.0
9000000004,1250.0,1250.0
9000000005,6000.0,6000.0
9000000006,-500.0,-500.0
9000000007,4000.0,4000.0
9000000008,2000.0,2000.0


## 5. Rules and odd values

In [79]:
# order 4500000003: line 000020 is rejected (abgru = 01)
vbap[vbap['vbeln'] == '4500000003']

,vbeln,posnr,matnr,kwmeng,meins,netpr,waerk,werks,pstyv,abgru,value
4,4500000003,000010,MAT00001,50.0,EA,25.0,EUR,DE01,TAN,None,1250.0
5,4500000003,000020,MAT00002,30.0,EA,42.0,EUR,DE01,TAN,01,1260.0


In [80]:
# rejected lines (abgru filled in)
vbap[vbap['abgru'].fillna('') != '']

,vbeln,posnr,matnr,kwmeng,meins,netpr,waerk,werks,pstyv,abgru,value
5,4500000003,000020,MAT00002,30.0,EA,42.0,EUR,DE01,TAN,01,1260.0


In [81]:
# deliveries of order 4500000003: only line 000010, so the rejected line 000020 was not shipped
lips[lips['vgbel'] == '4500000003']

,vbeln,posnr,matnr,lfimg,meins,werks,vgbel,vgpos
5,8000000004,000010,MAT00001,50.0,EA,DE01,4500000003,000010


In [82]:
# billing of order 4500000003: only line 000010, so the rejected line 000020 was not billed
vbrp[vbrp['aubel'] == '4500000003']

,vbeln,posnr,matnr,fkimg,meins,netwr,aubel,aupos,vgbel,vgpos
5,9000000004,000010,MAT00001,50.0,EA,1250.0,4500000003,000010,8000000004,000010


In [83]:
# credit memo line
vbrp[vbrp['netwr'] < 0]

,vbeln,posnr,matnr,fkimg,meins,netwr,aubel,aupos,vgbel,vgpos
7,9000000006,000010,MAT00001,20.0,EA,-500.0,4500000006,000010,9000000001,000010


In [84]:
# order division (spart) vs material group (matkl)
div = vbap.merge(vbak[['vbeln', 'spart']], on='vbeln')   # add the order's division
div = div.merge(mara[['matnr', 'matkl']], on='matnr')    # add the material's group
div[['vbeln', 'posnr', 'spart', 'matkl', 'value']]

,vbeln,posnr,spart,matkl,value
0,4500000001,000010,HA,HAIRPRO,2500.0
1,4500000001,000020,HA,HAIRPRO,900.0
2,4500000002,000010,AD,ADHESIV,3000.0
3,4500000002,000020,AD,ADHESIV,800.0
4,4500000003,000010,HA,HAIRPRO,1250.0
5,4500000003,000020,HA,HAIRPRO,1260.0
6,4500000004,000010,HA,HAIRPRO,3600.0
7,4500000004,000020,HA,ADHESIV,750.0
8,4500000005,000010,AD,ADHESIV,6000.0
9,4500000006,000010,HA,HAIRPRO,-500.0


In [85]:
# shipped vs ordered per order line (schema lines 186-187)
ordered = vbap.set_index(['vbeln', 'posnr'])['kwmeng']     # ordered quantity per order line
shipped = lips.groupby(['vgbel', 'vgpos'])['lfimg'].sum()  # shipped quantity per order line
pd.DataFrame({'ordered': ordered, 'shipped': shipped})     # empty shipped = not shipped

ordered  shipped
4500000001 000010    100.0    100.0
           000020     50.0     50.0
4500000002 000010    200.0    200.0
           000020    100.0    100.0
4500000003 000010     50.0     50.0
           000020     30.0      NaN
4500000004 000010    200.0      NaN
           000020     50.0      NaN
4500000005 000010    500.0    500.0
4500000006 000010     20.0      NaN
4500000007 000010    500.0    500.0
           000020    100.0    100.0
4500000008 000010     80.0     80.0

In [86]:
# does each line have the same currency as its order? (schema line 160)
both = vbap.merge(vbak[['vbeln', 'waerk']], on='vbeln', suffixes=('_line', '_order'))
int((both['waerk_line'] != both['waerk_order']).sum())   # number of lines that differ (0 = none)

0

In [87]:
# does each line use the material's base unit? (schema line 157)
both = vbap.merge(mara[['matnr', 'meins']], on='matnr', suffixes=('_line', '_material'))
int((both['meins_line'] != both['meins_material']).sum())   # number of lines that differ (0 = none)

0

In [88]:
# revenue date: billing date (fkdat) vs creation date (erdat) (schema line 214)
(vbrk['fkdat'] == vbrk['erdat']).all()   # True = same date on every billing document

np.True_

In [89]:
# codes found in the data
# the schema lists: auart TA/KR/RE (line 140), pstyv TAN/KRN/TANN (162), fkart F2/G2/S1 (209), mtart FERT/HALB/ROH (114)
pd.Series({
    'auart': vbak['auart'].unique(),
    'pstyv': vbap['pstyv'].unique(),
    'fkart': vbrk['fkart'].unique(),
    'mtart': mara['mtart'].unique(),
})

auart      [TA, KR]
pstyv    [TAN, KRN]
fkart      [F2, G2]
mtart        [FERT]
dtype: object

In [90]:
# MAT00001 billed quantity summed as it is (credit memo +20 instead of -20)
float(vbrp[vbrp['matnr'] == 'MAT00001']['fkimg'].sum())   # 250; with -20 it would be 210

250.0

In [91]:
# returns: orders of type RE (schema line 140) vs credit memos (G2)
pd.Series({'orders with auart RE': (vbak['auart'] == 'RE').sum(), 'credit memos (G2)': (vbrk['fkart'] == 'G2').sum()})

orders with auart RE    0
credit memos (G2)       1
dtype: int64

In [92]:
# billed value per currency (no exchange-rate table, so they can't be added)
vbrk.groupby('waerk')['netwr'].sum()

waerk
EUR    13950.0
USD     6000.0
Name: netwr, dtype: float64

In [93]:
# EUR billing lines with the order's division (spart) and the material's group (matkl)
eur = vbrp[vbrp['aubel'].isin(vbak[vbak['waerk'] == 'EUR']['vbeln'])]
eur = eur.merge(vbak[['vbeln', 'spart']].rename(columns={'vbeln': 'aubel'}), on='aubel')
eur = eur.merge(mara[['matnr', 'matkl']], on='matnr')

In [94]:
# EUR billed by order division
eur.groupby('spart')['netwr'].sum()

spart
AD     3800.0
HA    10150.0
Name: netwr, dtype: float64

In [95]:
# EUR billed by material group
eur.groupby('matkl')['netwr'].sum()

matkl
ADHESIV    7800.0
HAIRPRO    6150.0
Name: netwr, dtype: float64

## 6. Dates (today = 2026-05-04)

In [96]:
# first and last date of each date column
pd.DataFrame({
    'order created':   vbak['erdat'],
    'requested':       vbak['vdatu'],
    'goods issued':    likp['wadat_ist'],
    'billed':          vbrk['fkdat'],
    'open AR due':     bsid['faedt'],
    'cleared':         bsad['augdt'],
}).agg(['min', 'max'])   # columns have different lengths; min and max skip the gaps

,order created,requested,goods issued,billed,open AR due,cleared
min,2025-10-01,2025-10-15,2025-10-13,2025-10-14,2025-12-10,2025-11-05
max,2026-04-15,2026-05-01,2026-04-25,2026-04-26,2026-05-26,2025-12-10


In [97]:
# delivery lines with their goods issue date and their order's requested date
late = lips.merge(likp[['vbeln', 'wadat_ist']], on='vbeln')
late = late.merge(vbak[['vbeln', 'vdatu']], left_on='vgbel', right_on='vbeln')

In [98]:
# days late = goods issue date minus requested date (negative = early)
late['days_late'] = (late['wadat_ist'] - late['vdatu']).dt.days
late[['vgbel', 'vgpos', 'wadat_ist', 'vdatu', 'days_late']]

,vgbel,vgpos,wadat_ist,vdatu,days_late
0,4500000001,000010,2025-10-13,2025-10-15,-2
1,4500000001,000020,2025-10-13,2025-10-15,-2
2,4500000002,000010,2025-10-18,2025-10-20,-2
3,4500000002,000020,2025-10-18,2025-10-20,-2
4,4500000002,000010,2025-11-08,2025-10-20,19
5,4500000003,000010,2025-10-28,2025-10-25,3
6,4500000005,000010,2025-11-17,2025-11-20,-3
7,4500000007,000010,2026-01-29,2026-02-01,-3
8,4500000007,000020,2026-01-29,2026-02-01,-3
9,4500000008,000010,2026-04-25,2026-05-01,-6


In [99]:
# share on time (days_late <= 0), counted three ways
pd.Series({
    'delivery lines': (late['days_late'] <= 0).mean(),
    'order lines (last delivery)': (late.groupby(['vgbel', 'vgpos'])['days_late'].max() <= 0).mean(),
    'orders (last delivery)': (late.groupby('vgbel')['days_late'].max() <= 0).mean(),
}).round(3)

delivery lines                 0.800
order lines (last delivery)    0.778
orders (last delivery)         0.667
dtype: float64

In [100]:
# standard orders never shipped
backlog = vbak[(vbak['auart'] == 'TA') & (vbak['vbeln'].isin(lips['vgbel']) == False)].copy()
backlog['days_past_requested'] = (TODAY - backlog['vdatu']).dt.days
backlog[['vbeln', 'netwr', 'waerk', 'vdatu', 'days_past_requested']]

,vbeln,netwr,waerk,vdatu,days_past_requested
3,4500000004,4350.0,EUR,2026-04-01,33


In [101]:
# orders on time if the never-shipped, past-due order (backlog, cell above) counts as late
on_time_orders = (late.groupby('vgbel')['days_late'].max() <= 0).sum()
on_time_orders / (late['vgbel'].nunique() + len(backlog))   # 4 of 7 = 0.571

np.float64(0.5714285714285714)

In [102]:
# open receivables: days past due
bsid['days_overdue'] = (TODAY - bsid['faedt']).dt.days
bsid

,bukrs,kunnr,belnr,gjahr,buzei,bldat,faedt,wrbtr,waers,rebzg,days_overdue
0,EU01,C0000001,1400000003,2025,001,2025-11-10,2025-12-10,1500.0,EUR,9000000003,145
1,EU01,C0000005,1400000007,2026,001,2026-01-30,2026-03-31,4000.0,EUR,9000000007,34
2,EU01,C0000001,1400000008,2026,001,2026-04-26,2026-05-26,2000.0,EUR,9000000008,-22


## 7. Receivables and payments

In [103]:
# payment terms = due date minus posting date
ar['term_days'] = (ar['faedt'] - ar['bldat']).dt.days
ar[['belnr', 'kunnr', 'rebzg', 'term_days']]

,belnr,kunnr,rebzg,term_days
0,1400000003,C0000001,9000000003,30
1,1400000007,C0000005,9000000007,60
2,1400000008,C0000001,9000000008,30
3,1400000001,C0000001,9000000001,30
4,1400000002,C0000001,9000000002,30
5,1400000004,C0000003,9000000004,30
6,1400000005,C0000002,9000000005,30
7,1400000006,C0000001,9000000006,30


In [104]:
# cleared items: days between clearing and the due date (positive = early)
bsad['days_early'] = (bsad['faedt'] - bsad['augdt']).dt.days
bsad[['belnr', 'rebzg', 'wrbtr', 'augdt', 'days_early']]

,belnr,rebzg,wrbtr,augdt,days_early
0,1400000001,9000000001,3400.0,2025-11-05,8
1,1400000002,9000000002,2300.0,2025-11-15,3
2,1400000004,9000000004,1250.0,2025-11-20,8
3,1400000005,9000000005,6000.0,2025-12-10,8
4,1400000006,9000000006,-500.0,2025-11-05,16


In [105]:
# days to pay = clearing date minus billing date
pay = bsad.merge(vbrk[['vbeln', 'fkdat', 'fkart']], left_on='rebzg', right_on='vbeln')
pay['days_to_pay'] = (pay['augdt'] - pay['fkdat']).dt.days

In [106]:
# average days to pay: invoices only vs with the credit memo (cleared by offset)
pd.Series({'invoices only (F2)': pay[pay['fkart'] == 'F2']['days_to_pay'].mean(), 'with the credit memo': pay['days_to_pay'].mean()})

invoices only (F2)      23.25
with the credit memo    21.40
dtype: float64

In [107]:
# open receivables aged from posting date + 30 days instead of the due date
bsid['overdue_if_30_days'] = (TODAY - bsid['bldat'] - pd.Timedelta(days=30)).dt.days
bsid[['belnr', 'faedt', 'days_overdue', 'overdue_if_30_days']]

,belnr,faedt,days_overdue,overdue_if_30_days
0,1400000003,2025-12-10,145,145
1,1400000007,2026-03-31,34,64
2,1400000008,2026-05-26,-22,-22


In [108]:
# receivable amount vs the billing document's net value: equal on how many?
both = ar.merge(vbrk[['vbeln', 'netwr', 'mwsbp']], left_on='rebzg', right_on='vbeln')
int((both['wrbtr'] == both['netwr']).sum())   # 8 of 8: receivables hold the net amount

8

In [109]:
# open receivables: net (as stored) vs gross (net + tax)
open_items = both[both['augdt'].isnull()]
pd.Series({'net': open_items['wrbtr'].sum(), 'gross': (open_items['wrbtr'] + open_items['mwsbp']).sum()})

net      7500.0
gross    8925.0
dtype: float64

## 8. The 8 scenarios in the data file
Each scenario comment in `02_sample_data.sql` checked against the cells above.

| Scenario | Comment says | Data shows (section) | Match |
|---|---|---|---|
| 1. 4500000001 (lines 61-95) | on time, billed 3,400, paid before due date | goods issue 2 days early (6), billed 3,400 (4), cleared 8 days early (7) | ✓ |
| 2. 4500000002 (lines 97-152) | 2 deliveries, second late; 2,300 paid; 1,500 open, overdue | 2 deliveries on line 000010 (3), second 19 days late (6), 2,300 cleared (7), 1,500 open 145 days (6) | ✓ |
| 3. 4500000003 (lines 155-186) | line 000020 rejected, header 1,250, 3 days late | header 1,250 = active line (4), rejected line not shipped or billed (5), 3 days late (6) | ✓ |
| 4. 4500000004 (lines 189-207) | no delivery, 33 days past requested | never shipped, 33 days (6) | ✓ |
| 5. 4500000005 (lines 210-237) | USD, on time, 6,000 paid | 6,000 USD (2), 3 days early (6), cleared 8 days early (7) | ✓ |
| 6. 4500000006 (lines 240-268) | credit memo -500 for invoice 9000000001, cleared by offset (line 245) | KR -500 (2), credit memo points to 9000000001 (5), cleared 2025-11-05, the same day as 9000000001 (7); the offset partner is inferred, no clearing-document column | ✓ |
| 7. 4500000007 (lines 271-306) | delivered in full, only 4,000 billed, 4,200 not billed, 4,000 overdue | 4,200 not billed (3), billed 4,000 (4), 4,000 open 34 days (6), 60-day terms (7) | ✓ |
| 8. 4500000008 (lines 309-337) | on time, 2,000 open, not yet due | 6 days early (6), 2,000 open, due in 22 days (6) | ✓ |

## 9. Findings (what the analytics layer must handle)

| # | Finding | Where | Task 2 handles it by |
|---|---|---|---|
| 1 | order line 4500000002/000010 has 2 deliveries and 2 billing lines: joined on the order line it gives 4 rows, so 3,000 shows as 12,000 (billed 6,000). Schema line 47 warns only about deliveries | 3 | summing deliveries and billing per order line before joining |
| 2 | credit memo 9000000006: value -500 but quantity +20; schema line 225 says negative. Summed as is, MAT00001 billed quantity is 250, not 210 | 5 | billed quantity = -abs(`fkimg`) for G2 (S1 assumed); the KR order line keeps its source signs (20 x -25 = -500, data line 243) |
| 3 | the credit memo (1400000006, -500) is cleared by offset (data line 245), not paid: counted in days-to-pay it moves the average from 23.25 to 21.4 days. 23.25 covers the 4 paid invoices only: the 2 open overdue invoices (9000000003 and 9000000007, 175 and 94 days since billing) are not in it, so real payment speed is slower | 7 | days-to-pay on invoices (F2) only |
| 4 | due date is 60 days after posting on 1400000007 (C0000005), 30 on the other 7: aging from posting + 30 would show 64 days overdue, not 34 | 7 | aging from the due date `faedt` |
| 5 | receivables hold the net amount (`wrbtr` = `netwr` on 8 of 8, tax left out): open receivables are 7,500 net, 8,925 gross | 7 | reporting receivables as net; question for Finance |
| 6 | order division (`spart`) and material group (`matkl`) disagree on 2 lines (750 not shipped; 4,000 billed). EUR billed by `spart`: HA 10,150, AD 3,800; by `matkl`: HAIRPRO 6,150, ADHESIV 7,800 | 5 | keeping both; asking Sales which one is "the division" |
| 7 | EUR and USD, and no exchange-rate table: billed 13,950 EUR and 6,000 USD can't be added. Quantities in EA, KG and L can't be added either | 2, 5 | money per currency; quantities per material |
| 8 | the on-time rate depends on what you count: 80% of delivery lines, 77.8% of order lines, 66.7% of orders; 57.1% (4 of 7) if the unshipped, past-due order 4500000004 counts | 6 | on-time per order line, last goods issue vs `vdatu`; past-due unshipped lines shown separately |
| 9 | "returns" by order type finds none (0 orders with `auart` RE, schema line 140); the real return is the credit memo: -500 EUR, 20 EA (data line 241) | 5 | defining returns as G2 credit memos, not `auart` RE |

Checked and clean (0 found, shown in the cells above): duplicate keys on all 15 tables (section 2); broken links along the O2C chain, order to delivery to billing to receivable (section 3); a receivable in both open and cleared items (section 3); a billing document with no receivable (section 3).